# X0 - acquire the external data

Downloads the public inputs for the extended study, verifies them, **keeps the raw files on
Drive**, and writes one small CSV per source restricted to the **235 SH3 genes** of the frozen
atlas.

| Source | Size | Automated | Output |
|---|---|---|---|
| Domainome v5 tables (Zenodo) | 90 MB | yes | `domainome_sh3_variants.csv` |
| Domainome `files_to_reproduce_analyses` | 4.1 GB | yes, `PROFILE='full'` | energy models, gnomAD, ClinVar prep |
| ClinVar `variant_summary` | 450 MB | yes | `clinvar_sh3.csv` |
| gnomAD v4 (GraphQL API) | - | yes | `gnomad_sh3_missense.csv` |
| GRCh38 primary assembly | 882 MB | yes, `PROFILE='full'` | for G1 trinucleotide contexts |
| Atlas x external join | - | yes | `atlas_with_external.csv` |
| MGnify Stability | 5.8 GB | **no** - Google Form | upload by hand |
| dbNSFP v5.4a | ~50 GB | **no** - registration | upload by hand |
| COSMIC SBS signatures | 177 KB | **no** - terms | upload by hand |

**The frozen run is never modified.** The atlas and the B3 predictions are read only.

Everything is **resumable**: a file already on Drive with the right MD5 is not downloaded again,
and the gnomAD loop caches per gene. Safe to rerun after a disconnect.

### Runtime

X0 does **no GPU work** - it is network I/O and CSV parsing. The accelerator itself buys
nothing here, so the cheapest correct choice is **CPU with High-RAM on** (the ClinVar table is
about 4 GB uncompressed).

That said, a GPU runtime in Colab comes with a **bigger VM** - more vCPUs and more RAM - and
X0 *does* scale with those: the download pool and the pyarrow readers both use all cores. So if
compute units are not a concern, picking A100 will finish sooner. You are paying A100 rates for
I/O, and the saving is minutes; the 4.1 GB download and the copy to Drive are bandwidth-bound
either way.

The accelerator genuinely matters for **E3** (ESM3 / SaProt inference) and for any A2 re-run.

Worker counts adapt to whatever machine you get - the setup cell prints vCPU, RAM and GPU.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, time, random, shutil, hashlib, zipfile, tarfile, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd, numpy as np
try:
    import pyarrow.csv as pv          # much faster than the pandas C parser here
except ImportError:
    pv = None

# 'core' = the small, fast sources.  'full' = also the 4.1 GB Domainome archive
# and the 882 MB reference genome.
PROFILE = 'core'

# Parallelism scales with whatever machine Colab gave us. A GPU runtime is pointless
# for X0 itself (no tensor work) but it comes with more vCPUs and RAM, which does help.
NCPU = os.cpu_count() or 2
DL_WORKERS  = max(4, min(NCPU, 16))   # downloads: bandwidth- and CPU-bound
API_WORKERS = 3                       # gnomAD rate-limits hard; 8 triggers HTTP 429

import subprocess
try:
    import psutil; ram = f'{psutil.virtual_memory().total / 1e9:.0f} GB'
except Exception:
    ram = 'unknown'
try:    # absent on a CPU runtime, which raises rather than returning empty
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip() or 'none'
except (FileNotFoundError, OSError):
    gpu = 'none'
print(f'{NCPU} vCPU | {ram} RAM | GPU: {gpu}')
if gpu != 'none':
    print('  note: X0 does no GPU work. The GPU runtime only helps via its extra vCPU/RAM.')
    print('  Save the accelerator for E3 (ESM3 / SaProt inference).')
print(f'  downloads x{DL_WORKERS}, gnomAD x{API_WORKERS}')

DRIVE   = Path('/content/drive/MyDrive')
SCRATCH = Path('/content/raw')        # fast local disk; files are copied to Drive after

def pick(*names, what=''):
    """First existing folder among `names`. Tolerates renames (space vs underscore)."""
    for n in names:
        if (DRIVE / n).exists(): return DRIVE / n
    raise FileNotFoundError(
        f'No {what} folder in MyDrive. Tried: ' + ', '.join(names) +
        '. Create one or edit this cell.')

PROJECT = pick('sh3_repro/extended', 'SH3_extended', 'SH3 extended', 'sh3_extended',
               what='project')
FROZEN  = pick('sh3_repro', what='frozen-run')   # the 2026-09-22 run, READ ONLY

# Prefer the tidied layout; fall back to the old flat name if Drive still has it.
IN = next((PROJECT / n for n in ('input/external', 'input_new')
           if (PROJECT / n).exists()), PROJECT / 'input/external')
FROZEN_IN = PROJECT / 'input' / 'frozen'    # small read-only copies from sh3_repro
OUT  = PROJECT / 'results' / 'X0_external_data'
REF  = PROJECT / 'reference'
for p in (SCRATCH, OUT, REF,
          IN / 'domainome', IN / 'clinical', IN / 'genome', IN / 'stability',
          IN / 'signatures', IN / 'binding'):
    p.mkdir(parents=True, exist_ok=True)

print('profile :', PROFILE)
print('project :', PROJECT)
print('frozen  :', FROZEN)
!df -h /content | tail -1

In [ ]:
# --- the 235 genes -------------------------------------------------------
gene_file = REF / 'sh3_genes_uniprot.tsv'
atlas_tsv = FROZEN / 'results' / 'A0_cancer_atlas' / 'SH3_cancer_master.tsv'
if not atlas_tsv.exists():
    atlas_tsv = FROZEN_IN / 'results' / 'SH3_cancer_master.tsv'   # bundled copy

if gene_file.exists():
    genes = pd.read_csv(gene_file, sep='\t', header=None, names=['gene','uniprot'])
elif atlas_tsv.exists():
    genes = (pd.read_csv(atlas_tsv, sep='\t', usecols=['gene','uniprot'])
               .drop_duplicates().sort_values('gene'))
    genes.to_csv(gene_file, sep='\t', header=False, index=False)
else:
    raise FileNotFoundError('Need reference/sh3_genes_uniprot.tsv or the frozen atlas.')

GENE_SET, UNIPROT_SET = set(genes.gene), set(genes.uniprot)
print(f'{len(GENE_SET)} genes, {len(UNIPROT_SET)} UniProt accessions')

In [ ]:
# --- download helper: fetch to local disk, verify, then keep on Drive ----
def md5(path, chunk=1 << 22):
    h = hashlib.md5()
    with open(path, 'rb') as fh:
        for blk in iter(lambda: fh.read(chunk), b''): h.update(blk)
    return h.hexdigest()

MANIFEST = []

def fetch(url, dest, expect=None, label=''):
    """dest is the final path on Drive. Downloads via local disk (Drive FUSE is slow
    and unreliable for large writes), verifies, then copies. Returns the Drive path."""
    dest = Path(dest)
    if dest.exists():
        got = md5(dest)
        if expect is None or got == expect:
            print(f'  on Drive  {dest.name}  ({dest.stat().st_size/1e6:.1f} MB)')
            MANIFEST.append((label, dest.name, dest.stat().st_size, got, 'cached'))
            return dest
        print(f'  checksum mismatch on Drive copy of {dest.name} - refetching')

    tmp = SCRATCH / dest.name
    if not (tmp.exists() and (expect is None or md5(tmp) == expect)):
        print(f'  downloading {dest.name} ...')
        t0 = time.time()
        urllib.request.urlretrieve(url, tmp)
        print(f'    {tmp.stat().st_size/1e6:.1f} MB in {time.time()-t0:.0f}s')
    got = md5(tmp)
    if expect is not None and got != expect:
        raise ValueError(f'MD5 mismatch for {dest.name}: got {got}, expected {expect}')
    print(f'    copying to Drive ...')
    shutil.copy2(tmp, dest)
    verdict = 'verified' if expect else 'no published checksum'
    print(f'  ok        {dest.name}  {verdict}')
    MANIFEST.append((label, dest.name, dest.stat().st_size, got, verdict))
    return dest

## 1. Human Domainome 1.0

Zenodo **v5** (record 14356805), CC BY 4.0. v5 matches the Nature paper; v3 predates it and uses
different file names. Checksums are the ones Zenodo publishes, so these are truly verified.

Under `PROFILE='full'` this also takes `files_to_reproduce_analyses.tar.gz` (4.1 GB), which
contains the MoCHI energy models, the aPCA-to-kcal/mol conversion, the MegaScale-UniProt
mapping and the authors' gnomAD and ClinVar tables.

In [ ]:
ZEN = 'https://zenodo.org/records/14356805/files/'
DOM_CORE = {
 'Supplementary_Table_2_fitness_scores_normalized_domainranks.txt.zip': 'd0e744d6e50be563e46254ecc8586f24',
 'Supplementary_Table_5_aPCA_vs_variant_effect_predictors.txt.zip':     '2f3ff661e25aa5e6d92b664dc0709c17',
 'Supplementary_Table_1_library_design.txt':                            '9efcf1ae4f90e975f0c0719c79d136a6',
}
DOM_FULL = {
 'Supplementary_Table_4_homolog_predictions.txt.zip': '82d4c7212579e59f973fe87748c63c92',
 'files_to_reproduce_analyses.tar.gz':                '330ac11247bdccc7ea994c5f9d700048',
}

dom_dir = IN / 'domainome'
wanted = dict(DOM_CORE, **(DOM_FULL if PROFILE == 'full' else {}))

# Parallel: these are independent files, so bandwidth is the only limit.
with ThreadPoolExecutor(max_workers=min(DL_WORKERS, len(wanted))) as ex:
    futs = {ex.submit(fetch, ZEN + fn + '?download=1', dom_dir / fn, expect, 'domainome'): fn
            for fn, expect in wanted.items()}
    paths = {futs[f]: f.result() for f in as_completed(futs)}

for fn, p in paths.items():
    if fn.endswith('.zip'):
        with zipfile.ZipFile(p) as z: z.extractall(dom_dir)

t2 = next(dom_dir.rglob('*Table_2_fitness_scores_normalized_domainranks.txt'))
print('\ntable 2:', t2.name, f'{t2.stat().st_size/1e6:.0f} MB')

In [ ]:
# --- selective extract from the 4.1 GB archive (PROFILE='full') ----------
# The whole archive is 4.1 GB unpacked from a macOS path 7 levels deep; only these matter.
WANT = ['analysis_files/K50_dG_Dataset1_Dataset2_in_aPCA.csv',       # aPCA <-> kcal/mol
        'analysis_files/megascale_uniprot_mappings_offsets.tsv',     # MegaScale <-> UniProt
        'analysis_files/domains_from_tsuboyama_rocklin_overlaps.txt',# overlap / leakage check
        'analysis_files/gnomad_v4_variants_domainome.txt',
        'analysis_files/domainome_structural_annotation.txt',
        'analysis_files/variant_summary_jan24_SNVs_coding.txt',
        '02_validation.Rmd', '12_evaluate_mochi_models.Rmd']

tarball = dom_dir / 'files_to_reproduce_analyses.tar.gz'
if PROFILE == 'full' and tarball.exists():
    target = dom_dir / 'reproduce'; target.mkdir(exist_ok=True)
    with tarfile.open(tarball) as tf:
        members = [m for m in tf.getmembers()
                   if any(m.name.endswith(w) for w in WANT) and '/._' not in m.name]
        for m in members:
            m.name = Path(m.name).name          # flatten the macOS path
            tf.extract(m, target)
    print(f'extracted {len(members)} files to {target}')
    for f in sorted(target.iterdir()): print(f'  {f.stat().st_size/1e6:8.1f} MB  {f.name}')
else:
    print("skipped (PROFILE='core')")

In [ ]:
# --- SH3 rows only -------------------------------------------------------
# domain_ID is UNIPROT_PFAM_index. SH3 Pfam families: SH3_1, SH3_2, SH3_9.
SH3_PFAM = ('PF00018', 'PF07653', 'PF14604')

t0 = time.time()
if pv is not None:      # pyarrow reads this ~5x faster and uses all cores
    dom = pv.read_csv(t2, parse_options=pv.ParseOptions(delimiter='\t')).to_pandas()
else:
    dom = pd.read_csv(t2, sep='\t')
print(f'read {len(dom):,} rows in {time.time()-t0:.0f}s')
dom['STOP'] = dom.STOP.astype('string').str.upper()   # TRUE/FALSE may parse as bool
sh3 = dom[dom.domain_ID.str.contains('|'.join(SH3_PFAM))].copy()
sh3 = sh3[(sh3.STOP == 'FALSE') & sh3.normalized_fitness.notna()]

# Fix 1: 45 rows carry a fitness value but a null position/wt/mut. They cannot join.
n0 = len(sh3)
sh3 = sh3.dropna(subset=['position', 'wt_aa', 'mut_aa'])
sh3['position'] = sh3.position.astype(int)
print(f'dropped {n0 - len(sh3)} rows with a null key')

# Fix 2: one physical domain can carry two Pfam annotations at the same start
# (e.g. ITSN1 has both Q15811_PF00018_1003 and Q15811_PF07653_1003), so every
# variant in it appears twice. Collapse to the best-quality row per variant.
KEY = ['uniprot_ID', 'position', 'wt_aa', 'mut_aa']
ndup = sh3.duplicated(KEY, keep=False).sum()
sh3 = (sh3.sort_values('quality_rank')          # lower rank = better
          .drop_duplicates(KEY, keep='first')
          .sort_values(['uniprot_ID', 'position', 'mut_aa']))
print(f'collapsed {ndup:,} rows sharing a variant key -> {len(sh3):,} unique variants')
assert not sh3.duplicated(KEY).any()

sh3.to_csv(OUT / 'domainome_sh3_variants.csv', index=False)
print(f'{len(sh3):,} SH3 variants in {sh3.domain_ID.nunique()} domains, '
      f'{sh3.uniprot_ID.nunique()} proteins')
print(f'of those proteins, {len(set(sh3.uniprot_ID) & UNIPROT_SET)} are in the atlas')
del dom

## 2. ClinVar

`variant_summary.txt.gz`, ~450 MB, public domain, no login. NCBI reissues it monthly and
publishes no stable MD5, so the checksum is **recorded, not verified** - note the run date.

In [ ]:
CLINVAR_URL = 'https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz'
cv_path = fetch(CLINVAR_URL, IN / 'clinical' / 'variant_summary.txt.gz', None, 'clinvar')

AA3to1 = {'Ala':'A','Arg':'R','Asn':'N','Asp':'D','Cys':'C','Gln':'Q','Glu':'E','Gly':'G',
          'His':'H','Ile':'I','Leu':'L','Lys':'K','Met':'M','Phe':'F','Pro':'P','Ser':'S',
          'Thr':'T','Trp':'W','Tyr':'Y','Val':'V','Ter':'*'}
PROT_RE = re.compile(r'\(p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})\)')

keep = ['GeneSymbol','ClinicalSignificance','ReviewStatus','Name','Type','Assembly',
        'Chromosome','PositionVCF','ReferenceAlleleVCF','AlternateAlleleVCF','VariationID',
        'NumberSubmitters','PhenotypeList']
t0 = time.time()
chunks = []
if pv is not None:      # streaming pyarrow read: multi-core, low memory
    rdr = pv.open_csv(cv_path,
                      read_options=pv.ReadOptions(block_size=1 << 26),
                      parse_options=pv.ParseOptions(delimiter='\t'),
                      convert_options=pv.ConvertOptions(
                          include_columns=keep,
                          column_types={c: 'string' for c in keep}))
    for batch in rdr:
        ch = batch.to_pandas()
        ch = ch[(ch.Assembly == 'GRCh38') & ch.GeneSymbol.isin(GENE_SET)]
        if len(ch): chunks.append(ch)
else:
    for ch in pd.read_csv(cv_path, sep='\t', usecols=keep, dtype=str,
                          chunksize=500_000, low_memory=False):
        ch = ch[(ch.Assembly == 'GRCh38') & ch.GeneSymbol.isin(GENE_SET)]
        if len(ch): chunks.append(ch)
cv = pd.concat(chunks, ignore_index=True)
print(f'scanned ClinVar in {time.time()-t0:.0f}s')
print(f'{len(cv):,} GRCh38 ClinVar records in the 235 genes')

ex = cv.Name.str.extract(PROT_RE)
cv['wt_aa'] = ex[0].map(AA3to1)
cv['pos_protein'] = pd.to_numeric(ex[1], errors='coerce').astype('Int64')
cv['mut_aa'] = ex[2].map(AA3to1)
cv = cv[cv.wt_aa.notna() & cv.mut_aa.notna() & (cv.mut_aa != '*')]

STARS = {'no assertion criteria provided': 0, 'no assertion provided': 0,
         'no classification provided': 0, 'no classifications from unflagged records': 0,
         'criteria provided, single submitter': 1,
         'criteria provided, conflicting classifications': 1,
         'criteria provided, conflicting interpretations': 1,
         'criteria provided, multiple submitters, no conflicts': 2,
         'reviewed by expert panel': 3, 'practice guideline': 4}
cv['stars'] = cv.ReviewStatus.str.strip().str.lower().map(STARS).fillna(0).astype(int)
cv.to_csv(OUT / 'clinvar_sh3.csv', index=False)   # full record set, conflicts intact

# Fix 3: 249 keys carry conflicting submissions (e.g. ABL1 F311L is both Pathogenic
# with no assertion criteria and Uncertain from a single submitter). Collapse for the
# join only: highest star rating wins; on a tie, the more severe classification.
# This is a pre-registered rule, not a merge detail - change it deliberately.
SEVERITY = {'pathogenic': 5, 'likely pathogenic': 4, 'uncertain significance': 3,
            'likely benign': 2, 'benign': 1}
def sev(s):
    s = str(s).lower()
    return max((v for k, v in SEVERITY.items() if k in s), default=0)
cv['severity'] = cv.ClinicalSignificance.map(sev)

CVKEY = ['GeneSymbol', 'pos_protein', 'wt_aa', 'mut_aa']
nconf = cv.drop_duplicates(CVKEY + ['ClinicalSignificance']).duplicated(CVKEY, keep=False).sum()
cv_join = (cv.sort_values(['stars', 'severity'], ascending=False)
             .drop_duplicates(CVKEY, keep='first'))
print(f'{nconf} conflicting keys collapsed -> {len(cv_join):,} unique for the join')
assert not cv_join.duplicated(CVKEY).any()

print(f'{len(cv):,} missense records with a parsed protein change')
print(cv.ClinicalSignificance.value_counts().head(8).to_string())
print('\nstars:'); print(cv.stars.value_counts().sort_index().to_string())

## 3. gnomAD v4

The GraphQL API, one query per gene. gnomAD recommends this over the bulk VCFs for a small
number of records - the exome sites files total about 200 GB and you need ~19,000 codons.

5-15 minutes for 235 genes, resumable.

In [ ]:
API = 'https://gnomad.broadinstitute.org/api'
QUERY = ('query($g:String!){gene(gene_symbol:$g,reference_genome:GRCh38)'
         '{gene_id variants(dataset:gnomad_r4)'
         '{variant_id consequence hgvsp exome{ac an} genome{ac an}}}}')

import urllib.error

def gnomad_gene(sym, tries=5):
    """Returns (variants, error). error is None only on a genuine success, so a
    failure is never mistaken for 'this gene has no variants'."""
    body = json.dumps({'query': QUERY, 'variables': {'g': sym}}).encode()
    for k in range(tries):
        try:
            req = urllib.request.Request(API, data=body,
                                         headers={'Content-Type': 'application/json'})
            with urllib.request.urlopen(req, timeout=180) as r:
                g = (json.load(r).get('data') or {}).get('gene')
            return (g['variants'] if g else []), None
        except urllib.error.HTTPError as e:
            # 429 = rate limited. Back off long and hard, honouring Retry-After.
            wait = 60 * (k + 1) if e.code == 429 else 5 * (k + 1)
            ra = e.headers.get('Retry-After') if e.headers else None
            if ra:
                try: wait = max(wait, float(ra))
                except ValueError: pass
            if k == tries - 1: return None, f'HTTP {e.code}'
            time.sleep(wait + random.random() * 5)
        except Exception as e:
            if k == tries - 1: return None, type(e).__name__
            time.sleep(5 * (k + 1) + random.random())

cache_path = IN / 'clinical' / 'gnomad_cache.json'   # on Drive, survives a disconnect
cache = json.loads(cache_path.read_text()) if cache_path.exists() else {}

# An earlier version of this notebook cached failures as []. Every real gene has
# gnomAD variants, so an empty entry means a failed fetch: drop it and try again.
PURGE_EMPTY = True
if PURGE_EMPTY:
    dropped = [k for k, v in cache.items() if not v]
    for k in dropped: cache.pop(k)
    if dropped: print(f'dropped {len(dropped)} empty cache entries to refetch: {dropped[:8]}')

todo = [g for g in sorted(GENE_SET) if g not in cache]
print(f'{len(cache)} cached, {len(todo)} to fetch, {API_WORKERS} at a time')

t0, done, failed = time.time(), 0, []
with ThreadPoolExecutor(max_workers=API_WORKERS) as ex:
    futs = {ex.submit(gnomad_gene, s): s for s in todo}
    for f in as_completed(futs):
        sym = futs[f]
        variants, err = f.result()
        if err is None: cache[sym] = variants
        else: failed.append(sym); print(f'  ! {sym}: {err}')
        done += 1
        if done % 25 == 0 or done == len(todo):
            cache_path.write_text(json.dumps(cache))      # checkpoint to Drive
            print(f'  {done}/{len(todo)}  ({time.time()-t0:.0f}s, {len(failed)} failed)')
cache_path.write_text(json.dumps(cache))

In [ ]:
# --- retry whatever failed, slowly and one at a time ---------------------
for attempt in range(1, 4):
    if not failed: break
    print(f'retry pass {attempt}: {len(failed)} genes, sequential, 3s apart')
    still = []
    for sym in failed:
        variants, err = gnomad_gene(sym)
        if err is None: cache[sym] = variants
        else: still.append(sym)
        time.sleep(3)
    failed = still
    cache_path.write_text(json.dumps(cache))

missing = sorted(set(GENE_SET) - set(cache))
if missing:
    print(f'\nSTILL MISSING {len(missing)} genes - rerun this cell later: {missing}')
else:
    print(f'\nall {len(GENE_SET)} genes fetched')
empty = sorted(k for k in cache if not cache[k])
if empty: print(f'{len(empty)} genes genuinely returned no variants: {empty}')

In [ ]:
HGVSP_RE = re.compile(r'p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})')
rows = []
for sym, variants in cache.items():
    for v in variants or []:
        if v.get('consequence') != 'missense_variant' or not v.get('hgvsp'): continue
        m = HGVSP_RE.search(v['hgvsp'])
        if not m: continue
        ac = (v.get('exome') or {}).get('ac', 0) + (v.get('genome') or {}).get('ac', 0)
        an = (v.get('exome') or {}).get('an', 0) + (v.get('genome') or {}).get('an', 0)
        rows.append((sym, v['variant_id'], AA3to1.get(m.group(1)), int(m.group(2)),
                     AA3to1.get(m.group(3)), ac, an, ac / an if an else np.nan))

gn = pd.DataFrame(rows, columns=['gene','variant_id','wt_aa','pos_protein','mut_aa',
                                 'ac','an','af']).dropna(subset=['wt_aa','mut_aa'])
gn.to_csv(OUT / 'gnomad_sh3_missense.csv', index=False)
print(f'{len(gn):,} missense variants across {gn.gene.nunique()} genes')
print(f'  AF > 0.1%  : {(gn.af > 1e-3).sum():,}')
print(f'  singletons : {(gn.ac == 1).sum():,}')

## 4. GRCh38 reference genome (`PROFILE='full'`)

Needed by **G1** to read the trinucleotide context of each variant. Ensembl primary assembly,
882 MB gzipped - Ensembl chromosome naming (`1`, `2`, `X`) matches COSMIC, so no renaming.

Ensembl publishes a CHECKSUMS file rather than an MD5 per download, so this one is recorded.

In [ ]:
if PROFILE == 'full':
    FA = ('https://ftp.ensembl.org/pub/current_fasta/homo_sapiens/dna/'
          'Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz')
    fa_path = fetch(FA, IN / 'genome' / 'Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz',
                    None, 'genome')
    print('note: bgzip + samtools faidx it before random access:')
    print('  zcat ... | bgzip -c > genome.fa.bgz && samtools faidx genome.fa.bgz')
else:
    print("skipped (PROFILE='core')")

## 5. Join to the frozen atlas

Reads the B3 per-variant predictions and joins all three external sources onto them by
`uniprot / pos_protein / wt_aa / mut_aa`. Nothing in the frozen run is written to.

In [ ]:
pred_path = FROZEN / 'results' / 'B3_uncertainty_and_calls' / 'atlas_predictions.parquet'
if not pred_path.exists():
    print('B3 predictions not found - skipping the join. Fix FROZEN and rerun.')
else:
    pred = pd.read_parquet(pred_path)
    atlas = pd.read_csv(atlas_tsv, sep='\t', usecols=['variant_id','pos_protein'])
    pred = pred.merge(atlas, on='variant_id', how='left')

    # Fix 5: a left join must never change the row count. Every source below is
    # deduplicated on its join key first, and the assertion fails loudly if not.
    n = len(pred)
    def safe_merge(left, right, on, right_on=None, what=''):
        out = left.merge(right, left_on=on, right_on=right_on or on, how='left')
        assert len(out) == n, f'{what} merge expanded {n} -> {len(out)} rows'
        return out

    dcols = ['domain_ID','uniprot_ID','wt_aa','position','mut_aa',
             'normalized_fitness','normalized_fitness_sigma','quality_rank']
    j = safe_merge(pred, sh3[dcols], ['uniprot','pos_protein','wt_aa','mut_aa'],
                   ['uniprot_ID','position','wt_aa','mut_aa'], 'domainome')

    cvj = (cv_join[['GeneSymbol','pos_protein','wt_aa','mut_aa',
                    'ClinicalSignificance','stars','severity']]
             .rename(columns={'GeneSymbol':'gene','ClinicalSignificance':'clinvar_sig',
                              'stars':'clinvar_stars','severity':'clinvar_severity'}))
    j = safe_merge(j, cvj, ['gene','pos_protein','wt_aa','mut_aa'], None, 'clinvar')

    # Fix 4: gnomAD showed no duplicate protein changes when tested, but guard anyway -
    # if one aa change ever arrives via two alleles, sum the counts rather than duplicate.
    GKEY = ['gene','pos_protein','wt_aa','mut_aa']
    gnj = (gn.groupby(GKEY, as_index=False)
             .agg(gnomad_ac=('ac','sum'), gnomad_an=('an','max'),
                  gnomad_alleles=('variant_id','size')))
    gnj['gnomad_af'] = gnj.gnomad_ac / gnj.gnomad_an.replace(0, np.nan)
    j = safe_merge(j, gnj, GKEY, None, 'gnomad')

    j.to_csv(OUT / 'atlas_with_external.csv', index=False)
    j[j.normalized_fitness.notna()].to_csv(OUT / 'domainome_atlas_matched.csv', index=False)

    print(f'atlas variants          : {len(j):,}  (pred rows: {n:,} - must match)')
    print(f'  measured in Domainome : {j.normalized_fitness.notna().sum():,}')
    print(f'  in ClinVar            : {j.clinvar_sig.notna().sum():,}')
    print(f'  in gnomAD             : {j.gnomad_af.notna().sum():,}')

## 6. Manifest, checksums, results

In [ ]:
man = pd.DataFrame(MANIFEST, columns=['source','file','bytes','md5','verdict'])
man.to_csv(REF / 'data_manifest_X0.csv', index=False)
with open(REF / 'checksums_X0.md5', 'w') as fh:
    fh.write('# recorded by X0 on ' + time.strftime('%Y-%m-%d') + '\n')
    for _, r in man.iterrows(): fh.write(f'{r.md5}  {r.file}\n')

outs = sorted(OUT.glob('*.csv'))
metrics = {'date': time.strftime('%Y-%m-%d %H:%M'), 'profile': PROFILE,
           'genes': len(GENE_SET),
           'domainome_sh3_variants': int(len(sh3)),
           'domainome_sh3_domains': int(sh3.domain_ID.nunique()),
           'clinvar_records': int(len(cv)), 'gnomad_missense': int(len(gn)),
           'outputs': {p.name: p.stat().st_size for p in outs}}
(OUT / 'metrics_X0.json').write_text(json.dumps(metrics, indent=1))

lines = ['# X0 - external data acquisition', '',
         f'Run {metrics["date"]}, profile `{PROFILE}`. Genes: {len(GENE_SET)}.', '',
         '| Output | Rows | Size |', '|---|---|---|']
for p in outs:
    n = sum(1 for _ in open(p)) - 1
    lines.append(f'| `{p.name}` | {n:,} | {p.stat().st_size/1e6:.1f} MB |')
lines += ['', '## Downloads kept on Drive', '',
          '| Source | File | MD5 | Verdict |', '|---|---|---|---|']
for _, r in man.iterrows():
    lines.append(f'| {r.source} | `{r.file}` | `{r.md5}` | {r.verdict} |')
lines += ['', '## Not automated', '',
          'MGnify Stability (Google Form), dbNSFP v5.4a (registration), COSMIC SBS',
          'signatures (terms acceptance). Upload into `input/external/`. See `DOWNLOADS.md`.']
(OUT / 'RESULTS_X0.md').write_text('\n'.join(lines))
print('\n'.join(lines))

In [ ]:
print('kept on Drive:')
tot = 0
for p in sorted(IN.rglob('*')):
    if p.is_file():
        tot += p.stat().st_size
        print(f'  {p.stat().st_size/1e6:9.1f} MB  input_new/{p.relative_to(IN)}')
for p in sorted(OUT.glob('*')):
    tot += p.stat().st_size
    print(f'  {p.stat().st_size/1e6:9.1f} MB  results/X0_external_data/{p.name}')
print(f'\ntotal on Drive: {tot/1e9:.2f} GB')